In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║              CELL 1 — CẤU HÌNH & IMPORT                 ║
# ╚══════════════════════════════════════════════════════════╝
!pip install -q optuna

import os, json, time, gc, pickle, glob
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, backend as K
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['figure.dpi'] = 120
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, roc_auc_score,
    f1_score, accuracy_score, precision_recall_fscore_support
)
from sklearn.preprocessing import label_binarize
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Đường dẫn ──────────────────────────────────────────────
TRAIN_PATH = "/kaggle/input/datasets/rsnngo/37ft-thang3737/train_standard_p3.2.parquet"
VAL_PATH   = "/kaggle/input/datasets/rsnngo/37ft-thang3737/val_standard_p3.2.parquet"
TEST_PATH  = "/kaggle/input/datasets/rsnngo/37ft-thang3737/test_standard_p3.2.parquet"
OUTPUT_DIR = "/kaggle/working/"
TARGET_COL = "label"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Checkpoint Optuna cho Autoencoder Teacher ─────────────
OPTUNA_STUDY_AE_PATH = os.path.join(OUTPUT_DIR, 'optuna_study_autoencoder_teacher.pkl')

LABEL_NAMES = {
    0: 'Benign', 1: 'DDoS',  2: 'DoS',      3: 'BruteForce',
    4: 'Spoofing', 5: 'Recon', 6: 'Web-Based', 7: 'Mirai'
}

# ── Kiến trúc Autoencoder ────────────────────────────────
ENCODER_UNITS = [256, 128, 64, 32]   # bottleneck = 32
DECODER_UNITS = [64, 128, 256]       # đối xứng (không tính lớp cuối)
DROPOUT_RATE  = 0.3
AE_EPOCHS     = 30
AE_BATCH_SIZE = 256
AE_LR         = 1e-3

# ── Optuna cho classifier trên encoder ──────────────────
OPTUNA_N_TRIALS   = 20
OPTUNA_EPOCHS     = 30          # mỗi trial chỉ 30 epochs để tiết kiệm
OPTUNA_LR_LOW     = 1e-5
OPTUNA_LR_HIGH    = 1e-2
OPTUNA_BS_CHOICES = [64, 128, 256, 512]

# ── Train chính thức ────────────────────────────────────
FINAL_MAX_EPOCHS = 50
RANDOM_SEED      = 42

# ── Timeout ─────────────────────────────────────────────
TIMEOUT_S = 3600 * 12          # 12 giờ
T_START   = time.time()
DEADLINE  = T_START + TIMEOUT_S

# ── Seed ──────────────────────────────────────────────────
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print('=' * 58)
print('  AUTOENCODER TEACHER (Encoder + Classifier) — CICIoT2023 (8 classes)')
print('=' * 58)
print(f"  TensorFlow version  : {tf.__version__}")
print(f"  GPU available       : {tf.config.list_physical_devices('GPU')}")
print()
print('  [Kiến trúc Autoencoder]')
print(f'  Encoder units       : {ENCODER_UNITS}')
print(f'  Decoder units       : {DECODER_UNITS} (đối xứng)')
print(f'  Activation          : ReLU, BN, Dropout {DROPOUT_RATE}')
print('  Pretrain            : MSE loss, Adam, 30 epochs')
print('  Classifier head     : Dense(8, softmax) trên bottleneck')
print('  Optimizer           : Adam | loss: sparse_categorical_crossentropy')
print()
print(f'  n_trials            : {OPTUNA_N_TRIALS}')
print(f'  epoch/trial         : {OPTUNA_EPOCHS}')
print(f'  lr                  : log-uniform [{OPTUNA_LR_LOW}, {OPTUNA_LR_HIGH}]')
print(f'  batch_size          : categorical {OPTUNA_BS_CHOICES}')
print('  objective           : maximize val macro-F1')
print('  pruner              : MedianPruner(n_startup_trials=3, n_warmup_steps=5)')
print()
print(f'  Train chính thức    : {FINAL_MAX_EPOCHS} epoch (lưu best val loss)')
print('=' * 58)
print(f'  TIMEOUT             : {TIMEOUT_S/3600:.1f} giờ (deadline tính từ lúc kernel start)')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║              CELL 2 — LOAD DATA                         ║
# ║  Dữ liệu 2D cho MLP/Autoencoder                        ║
# ╚══════════════════════════════════════════════════════════╝
df_train = pd.read_parquet(TRAIN_PATH)
df_val   = pd.read_parquet(VAL_PATH)
df_test  = pd.read_parquet(TEST_PATH)

feature_cols = [c for c in df_train.columns if c != TARGET_COL]

X_train = df_train[feature_cols].values.astype(np.float32)
y_train = df_train[TARGET_COL].values.astype(np.int32)
X_val   = df_val[feature_cols].values.astype(np.float32)
y_val   = df_val[TARGET_COL].values.astype(np.int32)
X_test  = df_test[feature_cols].values.astype(np.float32)
y_test  = df_test[TARGET_COL].values.astype(np.int32)

NUM_CLASSES  = int(np.max(y_train)) + 1
NUM_FEATURES = X_train.shape[1]

# Lưu metadata
with open(f"{OUTPUT_DIR}metadata_ae.json", "w") as f:
    json.dump({"feature_cols": feature_cols, "target_col": TARGET_COL,
               "num_classes": NUM_CLASSES, "num_features": NUM_FEATURES}, f, indent=4)

print(f"Train : {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")
print(f"Num classes: {NUM_CLASSES} | Num features: {NUM_FEATURES}")
print("\nPhân bố lớp (train):")
for c in range(NUM_CLASSES):
    n = int((y_train == c).sum())
    print(f"  {c} {LABEL_NAMES.get(c, c):<11}: {n:>9,}  ({n/len(y_train)*100:5.2f}%)")

del df_train, df_val, df_test
gc.collect()

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║   CELL 3 — KIẾN TRÚC AUTOENCODER                       ║
# ║   Encoder: [256,128,64,32] + BN + ReLU + Dropout(0.3) ║
# ║   Decoder: đối xứng (64,128,256) + BN + ReLU + Dropout ║
# ║   Lớp cuối: Dense(input_dim, linear)                   ║
# ╚══════════════════════════════════════════════════════════╝

def build_autoencoder_and_encoder(input_dim, encoding_dim=32, dropout_rate=0.3):
    # Encoder
    inp = layers.Input(shape=(input_dim,), name='ae_input')
    x = inp
    for units in ENCODER_UNITS:
        x = layers.Dense(units)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.Dropout(dropout_rate)(x)
    # Layer bottleneck (đặt tên để dễ trích xuất)
    encoded = layers.Dense(encoding_dim, name='bottleneck')(x)
    # (Chú ý: bottleneck nằm sau cùng của encoder, trước khi vào decoder)

    # Decoder
    x = encoded
    for units in DECODER_UNITS:   # 64, 128, 256
        x = layers.Dense(units)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.Dropout(dropout_rate)(x)
    decoded = layers.Dense(input_dim, activation='linear', name='dec_output')(x)

    autoencoder = models.Model(inp, decoded, name='Autoencoder')
    encoder = models.Model(inp, encoded, name='Encoder')
    return autoencoder, encoder

# ── Kiểm tra ─────────────────────────────────────────────────
ae_model, enc_model = build_autoencoder_and_encoder(NUM_FEATURES)
ae_model.summary()
print(f"\nTổng params Autoencoder: {ae_model.count_params():,}")
del ae_model, enc_model
K.clear_session()
gc.collect()

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║   CELL 4 — PRETRAIN AUTOENCODER                         ║
# ║   Huấn luyện tái tạo (MSE) trên toàn bộ train+val      ║
# ║   Lưu cả autoencoder và encoder model                  ║
# ╚══════════════════════════════════════════════════════════╝

AE_PRETRAIN_PATH = os.path.join(OUTPUT_DIR, 'autoencoder_pretrained.keras')
ENCODER_PRETRAIN_PATH = os.path.join(OUTPUT_DIR, 'encoder_pretrained.keras')

print('=' * 58)
print('  PRETRAIN AUTOENCODER (Unsupervised)')
print(f'  Epochs: {AE_EPOCHS} | Batch: {AE_BATCH_SIZE} | LR: {AE_LR}')
print('=' * 58)

K.clear_session()
autoencoder, encoder = build_autoencoder_and_encoder(NUM_FEATURES)
autoencoder.compile(optimizer=tf.keras.optimizers.Adam(AE_LR), loss='mse')

checkpoint_ae = callbacks.ModelCheckpoint(
    AE_PRETRAIN_PATH,
    monitor='val_loss',
    mode='min',
    save_best_only=True,
    verbose=1
)

history_ae = autoencoder.fit(
    X_train, X_train,
    batch_size=AE_BATCH_SIZE,
    epochs=AE_EPOCHS,
    validation_data=(X_val, X_val),
    callbacks=[checkpoint_ae],
    verbose=1
)

# Load best autoencoder
autoencoder = tf.keras.models.load_model(AE_PRETRAIN_PATH)

# Lấy encoder từ autoencoder: tạo model từ input đến layer bottleneck
encoder = models.Model(autoencoder.input, autoencoder.get_layer('bottleneck').output)

# Lưu encoder model riêng để dùng sau
encoder.save(ENCODER_PRETRAIN_PATH)

best_ae_loss = min(history_ae.history['val_loss'])
print(f'\n✅ Pretrain done. Best val loss: {best_ae_loss:.6f}')
print(f'   Autoencoder saved: {AE_PRETRAIN_PATH}')
print(f'   Encoder model saved: {ENCODER_PRETRAIN_PATH}')

del autoencoder, encoder
K.clear_session()
gc.collect()

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║   CELL 5 — OPTUNA HYPERPARAMETER SEARCH                 ║
# ║   Tối ưu classifier head gắn trên encoder đã pretrain  ║
# ║   Sử dụng encoder model đã lưu                         ║
# ╚══════════════════════════════════════════════════════════╝

def build_classifier_on_encoder(encoder, num_classes, learning_rate, trainable_enc):
    encoder.trainable = trainable_enc
    inp = encoder.input
    x = encoder.output
    out = layers.Dense(num_classes, activation='softmax', name='class_output')(x)
    model = models.Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

class _OptunaPruneCallbackAE(tf.keras.callbacks.Callback):
    def __init__(self, trial, X_val, y_val, eval_bs=1024):
        super().__init__()
        self.trial = trial
        self.X_val = X_val
        self.y_val = y_val
        self.eval_bs = eval_bs
        self.last_f1 = 0.0

    def on_epoch_end(self, epoch, logs=None):
        val_proba = self.model.predict(self.X_val, batch_size=self.eval_bs, verbose=0)
        val_preds = np.argmax(val_proba, axis=1)
        macro_f1 = f1_score(self.y_val, val_preds, average='macro', zero_division=0)
        self.last_f1 = macro_f1
        self.trial.report(macro_f1, epoch + 1)
        if self.trial.should_prune():
            raise optuna.exceptions.TrialPruned()

def optuna_objective_ae(trial):
    lr = trial.suggest_float('lr', OPTUNA_LR_LOW, OPTUNA_LR_HIGH, log=True)
    bs = trial.suggest_categorical('batch_size', OPTUNA_BS_CHOICES)
    trainable_enc = trial.suggest_categorical('trainable_encoder', [True, False])

    # Tải encoder model đã pretrain (không phải weights)
    encoder = tf.keras.models.load_model(ENCODER_PRETRAIN_PATH)

    K.clear_session()
    model = build_classifier_on_encoder(encoder, NUM_CLASSES, lr, trainable_enc)

    prune_cb = _OptunaPruneCallbackAE(trial, X_val, y_val)

    try:
        model.fit(
            X_train, y_train,
            batch_size=bs,
            epochs=OPTUNA_EPOCHS,
            validation_data=(X_val, y_val),
            callbacks=[prune_cb],
            verbose=0
        )
        return prune_cb.last_f1
    finally:
        del model
        K.clear_session()
        gc.collect()

# ── Quản lý study với checkpoint (giữ nguyên) ─────────────
def _n_finished(st):
    return len([t for t in st.trials if t.state.is_finished()])

def _save_ckpt(st):
    tmp = OPTUNA_STUDY_AE_PATH + '.tmp'
    with open(tmp, 'wb') as f:
        pickle.dump(st, f)
    os.replace(tmp, OPTUNA_STUDY_AE_PATH)

def _find_input_checkpoint():
    for p in sorted(glob.glob('/kaggle/input/datasets/lamthicucb2203709/ae-ckpt/optuna_study_autoencoder_teacher.pkl', recursive=True)):
        try:
            with open(p, 'rb') as f:
                obj = pickle.load(f)
            if isinstance(obj, optuna.study.Study):
                return p, obj
        except Exception:
            continue
    return None, None

# ── Nạp study (giữ nguyên) ─────────────────────────────────
study_ae = None
if os.path.isfile(OPTUNA_STUDY_AE_PATH):
    with open(OPTUNA_STUDY_AE_PATH, 'rb') as f:
        study_ae = pickle.load(f)
    print(f'[Optuna AE] Resume từ OUTPUT: {OPTUNA_STUDY_AE_PATH}')
else:
    _p, _st = _find_input_checkpoint()
    if _st is not None:
        study_ae = _st
        print(f'[Optuna AE] Resume từ INPUT: {_p}')

if study_ae is None:
    sampler = optuna.samplers.TPESampler(seed=RANDOM_SEED)
    pruner  = optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=5)
    study_ae = optuna.create_study(direction='maximize',
                                   sampler=sampler, pruner=pruner)
    print(f'[Optuna AE] Tạo study MỚI — mục tiêu {OPTUNA_N_TRIALS} trials x {OPTUNA_EPOCHS} epoch/trial')

n_done      = _n_finished(study_ae)
n_remaining = max(0, OPTUNA_N_TRIALS - n_done)
print(f'[Optuna AE] Đã xong {n_done}/{OPTUNA_N_TRIALS} trials — cần chạy thêm {n_remaining}')

_save_ckpt(study_ae)

def _ckpt_deadline_cb(st, trial):
    _save_ckpt(st)
    left = DEADLINE - time.time()
    print(f'[Optuna AE] trial #{trial.number} {trial.state.name} — '
          f'{_n_finished(st)}/{OPTUNA_N_TRIALS} | còn {left/60:.1f} phút | checkpoint đã lưu')
    if time.time() >= DEADLINE:
        print('[Optuna AE] ⏰ Chạm DEADLINE → dừng sau trial này.')
        st.stop()

if n_remaining > 0:
    _budget = DEADLINE - time.time()
    if _budget > 0:
        t_optuna = time.time()
        study_ae.optimize(optuna_objective_ae,
                          n_trials=n_remaining,
                          timeout=_budget,
                          callbacks=[_ckpt_deadline_cb],
                          gc_after_trial=True,
                          show_progress_bar=True)
        print(f'[Optuna AE] Phiên này chạy {(time.time()-t_optuna)/60:.1f} phút')
    else:
        print('[Optuna AE] ⏰ Hết ngân sách thời gian trước khi kịp chạy trial nào.')

_save_ckpt(study_ae)
n_done = _n_finished(study_ae)
OPTUNA_AE_FINISHED = n_done >= OPTUNA_N_TRIALS
PIPELINE_CONTINUE_AE = OPTUNA_AE_FINISHED

print()
print('=' * 58)
if OPTUNA_AE_FINISHED:
    print(f'[Optuna AE] ✓ HOÀN TẤT {n_done}/{OPTUNA_N_TRIALS} trials')
    print(f'  Checkpoint: {OPTUNA_STUDY_AE_PATH}')
else:
    print(f'[Optuna AE] ⏸ DỪNG DO TIMEOUT — mới xong {n_done}/{OPTUNA_N_TRIALS} trials')
    print(f'  Checkpoint đã lưu: {OPTUNA_STUDY_AE_PATH}')
    print('  → Các bước tiếp theo:')
    print(f'    1. Tải {os.path.basename(OPTUNA_STUDY_AE_PATH)} từ Output của run này')
    print('    2. Upload làm dataset input (chỉ chứa 1 file .pkl này)')
    print('    3. Add dataset đó vào notebook rồi Run lại — study tự resume')
    print('  Các cell phía sau (viz / train / eval) sẽ TỰ ĐỘNG BỎ QUA ở run này.')
print('=' * 58)

if OPTUNA_AE_FINISHED:
    best_params_ae = study_ae.best_params
    best_val_macro_f1_ae = study_ae.best_value
    BEST_LR_AE         = best_params_ae['lr']
    BEST_BATCH_SIZE_AE = best_params_ae['batch_size']
    BEST_TRAINABLE_ENC = best_params_ae['trainable_encoder']

    with open(f"{OUTPUT_DIR}best_params_ae.json", "w") as f:
        json.dump(best_params_ae, f, indent=4)

    print()
    print('=' * 58)
    print('  KẾT QUẢ OPTUNA SEARCH (Autoencoder Teacher)')
    print('=' * 58)
    print(f'  Best val macro-F1 : {best_val_macro_f1_ae:.6f}')
    print(f'  lr                = {BEST_LR_AE:.7f}')
    print(f'  batch_size        = {BEST_BATCH_SIZE_AE}')
    print(f'  trainable_encoder = {BEST_TRAINABLE_ENC}')
    print('=' * 58)
    print('\nTrain chính thức sẽ dùng các tham số này.')

In [ ]:
# ── GUARD: chỉ chạy khi Optuna đã đủ trials ───────────────
if not globals().get('PIPELINE_CONTINUE_AE', True):
    print('⏭ BỎ QUA cell này — Optuna chưa đủ trials.')
else:
    # ── CELL 6 — TRỰC QUAN HÓA OPTUNA ─────────────────────
    completed_trials = [t for t in study_ae.trials if t.state == optuna.trial.TrialState.COMPLETE]
    pruned_trials    = [t for t in study_ae.trials if t.state == optuna.trial.TrialState.PRUNED]

    print(f'Completed trials : {len(completed_trials)}')
    print(f'Pruned trials    : {len(pruned_trials)}')
    print(f'Best trial       : #{study_ae.best_trial.number}  macro-F1={best_val_macro_f1_ae:.6f}')

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Optuna Hyperparameter Search — Autoencoder Teacher',
                 fontsize=14, fontweight='bold')

    # Optimization History
    ax = axes[0, 0]
    trial_numbers = [t.number for t in completed_trials]
    f1_values     = [t.value  for t in completed_trials]
    best_so_far   = [max(f1_values[:i+1]) for i in range(len(f1_values))]
    ax.scatter(trial_numbers, f1_values, c='steelblue', s=50, alpha=0.7, zorder=3, label='Trial macro-F1')
    ax.plot(trial_numbers, best_so_far, 'r-o', markersize=4, linewidth=1.5, label='Best so far', zorder=4)
    if pruned_trials:
        _pn = [t.number for t in pruned_trials if t.intermediate_values]
        _pv = [t.intermediate_values[max(t.intermediate_values)] for t in pruned_trials if t.intermediate_values]
        if _pn:
            ax.scatter(_pn, _pv, c='orange', marker='x', s=60, alpha=0.6, label='Pruned')
    ax.axhline(best_val_macro_f1_ae, color='red', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_xlabel('Trial number')
    ax.set_ylabel('Val Macro-F1')
    ax.set_title('Optimization History')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # Parameter Importance
    ax = axes[0, 1]
    param_names = ['lr', 'batch_size', 'trainable_encoder']
    importances = []
    for p in param_names:
        vals = np.array([t.params[p] for t in completed_trials], dtype=float)
        if p == 'lr':
            vals = np.log10(vals)
        elif p == 'batch_size':
            vals = np.array([OPTUNA_BS_CHOICES.index(int(v)) for v in vals], dtype=float)
        elif p == 'trainable_encoder':
            vals = vals.astype(float)  # True=1, False=0
        objs = np.array(f1_values, dtype=float)
        if len(vals) > 1 and np.std(vals) > 0 and np.std(objs) > 0:
            corr = abs(np.corrcoef(vals, objs)[0, 1])
        else:
            corr = 0.0
        importances.append(0.0 if np.isnan(corr) else corr)

    tot = sum(importances)
    importances = [i / tot for i in importances] if tot > 0 else [0.0] * len(param_names)
    bars = ax.barh(param_names, importances, color=['steelblue', 'coral', 'green'])
    ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=9)
    ax.set_xlabel('Relative importance (|correlation|, normalized)')
    ax.set_title('Parameter Importance')
    ax.set_xlim(0, max(importances + [0.1]) * 1.25)
    ax.grid(True, alpha=0.3, axis='x')

    # Parallel coordinate
    ax = axes[1, 0]
    if completed_trials:
        lrs = np.log10([t.params['lr'] for t in completed_trials])
        bss = np.array([OPTUNA_BS_CHOICES.index(int(t.params['batch_size'])) for t in completed_trials], dtype=float)
        tens = np.array([1.0 if t.params['trainable_encoder'] else 0.0 for t in completed_trials], dtype=float)
        f1s = np.array(f1_values)
        def _norm(a):
            rng = a.max() - a.min()
            return (a - a.min()) / rng if rng > 0 else np.full_like(a, 0.5)
        cols = [_norm(lrs), _norm(bss), _norm(tens), _norm(f1s)]
        xs = [0, 1, 2, 3]
        cmap = plt.cm.viridis
        f1n = _norm(f1s)
        for i in range(len(completed_trials)):
            ax.plot(xs, [c[i] for c in cols], '-o', markersize=3, color=cmap(f1n[i]), alpha=0.7, linewidth=1)
        ax.set_xticks(xs)
        ax.set_xticklabels(['log10(lr)', 'batch_size', 'trainable_enc', 'macro-F1'])
        ax.set_ylabel('Normalized value')
        ax.set_title('Parallel Coordinate (color = macro-F1)')
        ax.grid(True, alpha=0.3)

    # Scatter lr x batch_size
    ax = axes[1, 1]
    if completed_trials:
        sc = ax.scatter(np.log10([t.params['lr'] for t in completed_trials]),
                        [t.params['batch_size'] for t in completed_trials],
                        c=f1_values, cmap='viridis', s=110, edgecolors='k', linewidths=0.5)
        bt = study_ae.best_trial
        ax.scatter(np.log10(bt.params['lr']), bt.params['batch_size'],
                   marker='*', s=420, c='red', edgecolors='k', linewidths=0.8, label='Best trial', zorder=5)
        plt.colorbar(sc, ax=ax, label='Val Macro-F1')
        ax.set_xlabel('log10(learning rate)')
        ax.set_ylabel('batch_size')
        ax.set_yscale('log', base=2)
        ax.set_yticks(OPTUNA_BS_CHOICES)
        ax.set_yticklabels(OPTUNA_BS_CHOICES)
        ax.set_title('Search Space Coverage')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    optuna_plot_path = os.path.join(OUTPUT_DIR, 'optuna_search_ae.png')
    plt.savefig(optuna_plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  Optuna plots saved: {optuna_plot_path}')

    # Bảng tóm tắt tất cả trials
    print('\n' + '=' * 90)
    print('  TẤT CẢ TRIALS (Autoencoder Teacher)')
    print('=' * 90)
    print(f"  {'#':>3}  {'state':<9} {'lr':>12} {'batch':>7} {'trainable_enc':>15} {'macro-F1':>11}")
    print('-' * 90)
    for t in study_ae.trials:
        v    = f'{t.value:.6f}' if t.value is not None else '—'
        lr_s = f"{t.params['lr']:.3e}" if 'lr' in t.params else '—'
        bs_s = t.params.get('batch_size', '—')
        te_s = str(t.params.get('trainable_encoder', '—'))
        star = '  ←BEST' if (t.state == optuna.trial.TrialState.COMPLETE and t.number == study_ae.best_trial.number) else ''
        print(f'  {t.number:>3}  {t.state.name:<9} {lr_s:>12} {str(bs_s):>7} {te_s:>15} {v:>11}{star}')
    print('=' * 90)

In [ ]:
# ── GUARD ─────────────────────────────────────────────────
if not globals().get('PIPELINE_CONTINUE_AE', True):
    print('⏭ BỎ QUA cell này — Optuna chưa đủ trials.')
else:
    # ── CELL 7 — TRAIN CHÍNH THỨC ─────────────────────────
    with open(f"{OUTPUT_DIR}best_params_ae.json", "r") as f:
        best_params_ae = json.load(f)
    BEST_LR_AE         = best_params_ae['lr']
    BEST_BATCH_SIZE_AE = best_params_ae['batch_size']
    BEST_TRAINABLE_ENC = best_params_ae['trainable_encoder']

    TEACHER_MODEL_PATH = f"{OUTPUT_DIR}teacher_autoencoder_best.keras"

    print('=' * 58)
    print('  TRAIN CHÍNH THỨC — AUTOENCODER TEACHER')
    print(f'  Batch={BEST_BATCH_SIZE_AE}, LR={BEST_LR_AE:.6f}')
    print(f'  Trainable encoder = {BEST_TRAINABLE_ENC}')
    print(f'  MaxEpoch={FINAL_MAX_EPOCHS}  (KHÔNG EarlyStopping)')
    print('  Lưu best epoch theo val loss thấp nhất')
    print('=' * 58)

    # Tải encoder model đã pretrain
    encoder = tf.keras.models.load_model(ENCODER_PRETRAIN_PATH)

    K.clear_session()
    tf.random.set_seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)

    model = build_classifier_on_encoder(encoder, NUM_CLASSES, BEST_LR_AE, BEST_TRAINABLE_ENC)
    print('  Model khởi tạo (encoder pretrained + classification head).')

    checkpoint = callbacks.ModelCheckpoint(
        TEACHER_MODEL_PATH,
        monitor='val_loss',
        mode='min',
        save_best_only=True,
        verbose=1
    )

    start_time = time.time()
    history = model.fit(
        X_train, y_train,
        batch_size=BEST_BATCH_SIZE_AE,
        epochs=FINAL_MAX_EPOCHS,
        validation_data=(X_val, y_val),
        callbacks=[checkpoint],
        verbose=1
    )
    train_time = time.time() - start_time

    hist = history.history
    best_epoch    = int(np.argmin(hist['val_loss'])) + 1
    best_val_loss = float(np.min(hist['val_loss']))

    model = tf.keras.models.load_model(TEACHER_MODEL_PATH)

    with open(f"{OUTPUT_DIR}train_time_ae.txt", "w") as f:
        f.write(str(train_time))

    print(f'\n==> Training done. {train_time:.1f}s ({train_time/60:.1f} phút)')
    print(f'    Best epoch : {best_epoch}/{FINAL_MAX_EPOCHS}  |  Best val loss : {best_val_loss:.6f}')
    print(f'    Model tốt nhất được lưu tại: {TEACHER_MODEL_PATH}')

In [ ]:
# ── GUARD ─────────────────────────────────────────────────
if not globals().get('PIPELINE_CONTINUE_AE', True):
    print('⏭ BỎ QUA cell này — Optuna chưa đủ trials.')
else:
    # ── CELL 8 — Biểu đồ training curves ──────────────────
    hist = history.history
    epochs_range = range(1, len(hist['loss']) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(epochs_range, hist['loss'], label='Train Loss', color='steelblue')
    axes[0].plot(epochs_range, hist['val_loss'], label='Val Loss', color='orange')
    axes[0].axvline(best_epoch, color='red', linestyle='--', linewidth=1.2,
                    label=f'Best epoch ({best_epoch})  val={best_val_loss:.4f}')
    axes[0].set_title('Autoencoder Teacher - Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True)

    axes[1].plot(epochs_range, hist['accuracy'], label='Train Accuracy', color='steelblue')
    axes[1].plot(epochs_range, hist['val_accuracy'], label='Val Accuracy', color='orange')
    axes[1].axvline(best_epoch, color='red', linestyle='--', linewidth=1.2,
                    label=f'Best epoch ({best_epoch})')
    axes[1].set_title('Autoencoder Teacher - Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}training_curves_ae.png", dpi=150)
    plt.show()
    print("✅ Đã lưu biểu đồ training curves.")

In [ ]:
# ── GUARD ─────────────────────────────────────────────────
if not globals().get('PIPELINE_CONTINUE_AE', True):
    print('⏭ BỎ QUA cell này — Optuna chưa đủ trials.')
else:
    # ── CELL 9 — Đánh giá trên tập test ──────────────────
    best_model = tf.keras.models.load_model(TEACHER_MODEL_PATH)

    y_proba = best_model.predict(X_test, batch_size=128, verbose=1)
    y_pred = np.argmax(y_proba, axis=1)

    report = classification_report(y_test, y_pred, digits=4, zero_division=0)
    print("===== Classification Report (Autoencoder Teacher) =====")
    print(report)
    with open(f"{OUTPUT_DIR}classification_report_ae.txt", "w") as f:
        f.write(report)

    acc = accuracy_score(y_test, y_pred)
    prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(
        y_test, y_pred, average='macro', zero_division=0)
    prec_weighted, rec_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_test, y_pred, average='weighted', zero_division=0)

    print(f"\nAccuracy      : {acc:.4f}")
    print(f"Macro F1      : {f1_macro:.4f}")
    print(f"Weighted F1   : {f1_weighted:.4f}")

    metrics = {
        'accuracy': acc,
        'macro_f1': f1_macro,
        'weighted_f1': f1_weighted,
        'macro_precision': prec_macro,
        'macro_recall': rec_macro,
        'best_epoch': best_epoch,
        'best_val_loss': best_val_loss,
        'best_lr': BEST_LR_AE,
        'best_batch_size': BEST_BATCH_SIZE_AE,
        'best_trainable_encoder': BEST_TRAINABLE_ENC,
        'optuna_best_val_macro_f1': float(best_val_macro_f1_ae),
    }
    with open(f"{OUTPUT_DIR}test_metrics_ae.json", "w") as f:
        json.dump(metrics, f, indent=4)

    # ── ROC Curves & AUC ────────────────────────────────────
    y_test_bin = label_binarize(y_test, classes=range(NUM_CLASSES))
    plt.figure(figsize=(10, 7))
    colors = plt.cm.tab10(np.linspace(0, 1, NUM_CLASSES))
    auc_scores = []
    for i, color in enumerate(colors):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
        auc_val = roc_auc_score(y_test_bin[:, i], y_proba[:, i])
        auc_scores.append(auc_val)
        plt.plot(fpr, tpr, color=color, lw=1.5,
                 label=f'Class {i} {LABEL_NAMES.get(i, "")} (AUC={auc_val:.4f})')
    macro_auc = np.mean(auc_scores)
    plt.plot([0, 1], [0, 1], 'k--', lw=1)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curves — Autoencoder Teacher (Macro AUC = {macro_auc:.4f})')
    plt.legend(loc='lower right', fontsize=8)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}roc_curves_ae.png", dpi=150)
    plt.show()
    print(f"Macro AUC: {macro_auc:.4f}")
    with open(f"{OUTPUT_DIR}macro_auc_ae.txt", "w") as f:
        f.write(str(macro_auc))

In [ ]:
# ── GUARD ─────────────────────────────────────────────────
if not globals().get('PIPELINE_CONTINUE_AE', True):
    print('⏭ BỎ QUA cell này — Optuna chưa đủ trials.')
else:
    # ── CELL 10 — Confusion Matrix và Heaviness Metrics ──
    cm = confusion_matrix(y_test, y_pred)
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

    annot = np.empty_like(cm, dtype=object)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            annot[i, j] = f"{cm[i,j]:,}\n({cm_pct[i,j]:.1f}%)"

    plt.figure(figsize=(12, 10))
    sns.heatmap(cm_pct, annot=annot, fmt='', cmap='Blues',
                xticklabels=range(NUM_CLASSES),
                yticklabels=range(NUM_CLASSES),
                vmin=0, vmax=100)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title('Confusion Matrix (counts and row percentages) — Autoencoder Teacher')
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}confusion_matrix_ae.png", dpi=150)
    plt.show()
    print("✅ Đã lưu confusion matrix.")

    # ── Heaviness Metrics ──────────────────────────────────
    model_path = TEACHER_MODEL_PATH
    size_bytes = os.path.getsize(model_path)
    size_mb = size_bytes / (1024 * 1024)

    total_params = best_model.count_params()
    trainable_params = sum(tf.keras.backend.count_params(w) for w in best_model.trainable_weights)

    for _ in range(5):
        best_model.predict(X_test[:100], batch_size=128, verbose=0)
    N = 2000
    t0 = time.perf_counter()
    best_model.predict(X_test[:N], batch_size=128, verbose=0)
    inf_time_per_sample = (time.perf_counter() - t0) * 1000 / N

    heaviness = {
        "model_size_mb": round(size_mb, 6),
        "total_params": int(total_params),
        "trainable_params": int(trainable_params),
        "train_time_s": round(train_time, 4),
        "inference_ms_per_sample": round(inf_time_per_sample, 6)
    }
    print("===== Heaviness Metrics =====")
    for k, v in heaviness.items():
        print(f"  {k:<25s}: {v}")
    with open(f"{OUTPUT_DIR}heaviness_ae.json", "w") as f:
        json.dump(heaviness, f, indent=4)

    # ── Tổng kết file đã xuất ─────────────────────────────
    print()
    print('=' * 58)
    print('  FILE ĐÃ XUẤT TRONG', OUTPUT_DIR)
    print('=' * 58)
    for fn in sorted(os.listdir(OUTPUT_DIR)):
        fp = os.path.join(OUTPUT_DIR, fn)
        if os.path.isfile(fp):
            print(f"  {fn:<40} {os.path.getsize(fp)/1024:>10.1f} KB")
    print('=' * 58)